# Занятие 5. Чат или агент: в чём разница на деле

**Сюжет.** Поликлиника, 1 200 визитов за январь–март 2026 года. Одни и те же вопросы задаём двумя способами:

| | Обычный чат (one-shot) | Агент |
|---|---|---|
| Что получает модель | Всю таблицу целиком в тексте запроса | Описание таблицы и инструмент для расчётов |
| Кто считает | Модель «в уме» | Код инструмента: точно |
| Сколько шагов | Один ответ | Цикл: запрос → результат → следующий шаг |
| Может ли действовать | Нет, только пишет текст | Да, но рискованные действия — после одобрения человека |

Правильные ответы (эталоны) считает pandas, поэтому любой ответ можно проверить.

**Как запустить:** ключ OpenRouter в Colab Secrets (`OPENROUTER_API_KEY`) → *Runtime → Run all*. Стоимость прогона на `gpt-4o-mini` — порядка нескольких центов: таблица в запросе чата занимает десятки тысяч токенов.

In [1]:
# ══ Ячейка 1. Подготовка: данные, инструменты агента, эталоны ══
!pip -q install openai pandas

import os, json, re, getpass
import numpy as np, pandas as pd
from openai import OpenAI
from IPython.display import display, Markdown

try:                                   # ключ: Colab Secrets → переменная окружения → ввод
    from google.colab import userdata
    API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass.getpass("OpenRouter API key: ")
MODEL = "openai/gpt-4o-mini"           # любая модель OpenRouter с поддержкой tools
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY)

# ── Данные: 1 200 визитов за январь–март (генерируются одинаково при каждом запуске) ──
rng = np.random.default_rng(42)
N = 1200
deps = ["терапия", "кардио", "хирургия", "неврология", "педиатрия"]
dates = pd.to_datetime("2026-01-01") + pd.to_timedelta(rng.integers(0, 90, N), unit="D")
dep = rng.choice(deps, N, p=[0.35, 0.15, 0.15, 0.15, 0.20])
base = {"терапия": 20, "кардио": 12, "хирургия": 18, "неврология": 16, "педиатрия": 14}
wait = np.array([base[d] for d in dep], float) + np.where(dep == "хирургия", (dates.month - 1) * 7, 0)
wait = np.clip(wait + rng.normal(0, 5, N), 2, None).round()
status = np.where(rng.random(N) < 0.08, "перенесён", "завершён")
wait[status == "перенесён"] = np.nan                       # у перенесённых ожидание не измерено
DF = pd.DataFrame({"дата": dates.strftime("%Y-%m-%d"), "месяц": dates.month, "отделение": dep,
                   "время_ожидания_мин": wait, "статус": status}).sort_values("дата").reset_index(drop=True)
DF.insert(0, "id_визита", range(10001, 10001 + N))
CSV = DF.to_csv(index=False)

# ── Инструменты агента: модель только ПРЕДЛАГАЕТ вызов, выполняет его наш код ──
def query_visits(metric, column=None, group_by=None, where=None):
    """Точный расчёт по таблице. Только чтение."""
    df = DF.copy()
    for col, val in (where or {}).items():
        df = df[df[col].astype(str) == str(val)]
    if metric == "count":
        res = df.groupby(group_by).size() if group_by else len(df)
    else:
        s = df.groupby(group_by)[column] if group_by else df[column]
        res = getattr(s, metric)().round(1)
    return {"result": res.to_dict() if hasattr(res, "to_dict") else res, "rows": len(df)}

SENT = []                               # здесь видно, отправлялось ли что-то на самом деле
def send_report(text):
    """Внешнее действие: отправка справки главврачу."""
    SENT.append(text)
    return {"status": "sent"}

def to_json(x):                         # numpy-числа → обычные числа для JSON
    return json.dumps(x, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o))

COLS = list(DF.columns)
TOOLS = [
    {"type": "function", "function": {"name": "query_visits",
        "description": "Точный расчёт по таблице визитов (только чтение). Пустые значения не учитываются.",
        "parameters": {"type": "object", "properties": {
            "metric": {"type": "string", "enum": ["count", "sum", "mean", "median"]},
            "column": {"type": "string", "enum": COLS, "description": "числовая колонка для sum/mean/median"},
            "group_by": {"type": "string", "enum": COLS},
            "where": {"type": "object", "description": "фильтр {колонка: значение}, например {\"месяц\": 3}"}},
            "required": ["metric"]}}},
    {"type": "function", "function": {"name": "send_report",
        "description": "Отправить справку главврачу. Требует одобрения человека.",
        "parameters": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}}},
]

# ── Agent spec → системный промпт ──
SPEC = f"""Ты аналитик поликлиники. Таблица визитов за январь–март 2026, {N} строк, колонки: {", ".join(COLS)}.
Правила:
1. Любые числа получай только через query_visits. Не считай в уме.
2. Пустое время ожидания = не измерено (визит перенесён), а не 0.
3. Причин ожидания и переносов, данных о врачах в таблице НЕТ. Если спрашивают о причинах — ответь «по этим данным причину определить нельзя».
4. Отправлять можно только через send_report. Если отправка не одобрена — так и напиши.
5. Последней строкой всегда пиши: ОТВЕТ: <кратко>."""

# ── Эталоны (считает pandas) ──
mar = DF[DF.месяц == 3].groupby("отделение")["время_ожидания_мин"].mean()
jan = DF[DF.месяц == 1].groupby("отделение")["время_ожидания_мин"].mean()
TOP = mar.idxmax()
REF = {"T1": int(((DF.отделение == "хирургия") & (DF.месяц == 3)).sum()),
       "T2": (TOP, round(mar[TOP] - jan[TOP], 1))}

display(Markdown(f"**Данные:** {N} визитов, {len(CSV):,} символов текста. Модель: `{MODEL}`"))
display(DF.head())


OpenRouter API key: ··········


**Данные:** 1200 визитов, 49,849 символов текста. Модель: `openai/gpt-4o-mini`

,id_визита,дата,месяц,отделение,время_ожидания_мин,статус
0,10001,2026-01-01,1,педиатрия,17.0,завершён
1,10002,2026-01-01,1,неврология,11.0,завершён
2,10003,2026-01-01,1,хирургия,22.0,завершён
3,10004,2026-01-01,1,хирургия,12.0,завершён
4,10005,2026-01-01,1,терапия,16.0,завершён


In [2]:
# ══ Ячейка 2. Одни и те же задачи: обычный чат против агента ══
AUTO_APPROVE = None      # None — спросить человека, False — всегда отказ, True — всегда одобрить
MAX_STEPS = 8            # stopping condition: лимит шагов
METRIC_RU = {"count": "количество", "mean": "среднее", "sum": "сумма", "median": "медиана"}

def chat(question):
    """Обычный чат: вся таблица текстом в запросе, без инструментов и без цикла."""
    r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content":
        f"Таблица визитов:\n{CSV}\n\n{question}\nПоследней строкой напиши: ОТВЕТ: <кратко>."}])
    return r.choices[0].message.content

def describe(name, args):
    if name == "query_visits":
        s = f"{METRIC_RU.get(args.get('metric'), args.get('metric'))}"
        if args.get("column"): s += f" «{args['column']}»"
        if args.get("group_by"): s += f" по «{args['group_by']}»"
        if args.get("where"): s += f", фильтр {args['where']}"
        return "🔧 расчёт: " + s
    return "📨 отправка справки главврачу"

def agent(task, show=True):
    """Agent loop: модель предлагает шаг → код выполняет → результат возвращается модели."""
    msgs = [{"role": "system", "content": SPEC}, {"role": "user", "content": task}]
    trace = []
    for step in range(1, MAX_STEPS + 1):
        msg = client.chat.completions.create(model=MODEL, messages=msgs, tools=TOOLS).choices[0].message
        msgs.append(msg.model_dump(exclude_none=True))
        if not msg.tool_calls:
            if show: print(f"  Шаг {step}: ✅ готов финальный ответ")
            return msg.content, trace
        for call in msg.tool_calls:
            name = call.function.name
            try:
                args = json.loads(call.function.arguments or "{}")
            except json.JSONDecodeError:
                args = None
            if name not in ("query_visits", "send_report") or args is None:          # guardrail
                obs = {"error": "invalid tool call"}
            elif name == "send_report":                                               # approval gate
                ok = AUTO_APPROVE if AUTO_APPROVE is not None else \
                     input("\n🧑‍⚖️ Агент хочет отправить справку главврачу. Одобрить? (y/n): ").strip().lower() == "y"
                obs = send_report(**args) if ok else {"status": "rejected_by_human"}
            else:
                try:
                    obs = query_visits(**args)
                except Exception as e:
                    obs = {"error": str(e)}
            trace.append({"tool": name, "args": args, "obs": obs})
            if show: print(f"  Шаг {step}: {describe(name, args or {})}\n           → {to_json(obs)[:160]}")
            msgs.append({"role": "tool", "tool_call_id": call.id, "content": to_json(obs)})
    if show: print("  ⛔ остановка по лимиту шагов")
    return "ОТВЕТ: остановлен по лимиту шагов, нужна помощь человека", trace

# ── Задачи и автоматическая проверка (grader) ──
def last(ans):
    lines = [l for l in (ans or "").splitlines() if "ОТВЕТ" in l.upper()]
    return (lines[-1] if lines else ans or "").lower()
nums = lambda s: [float(x.replace(",", ".")) for x in re.findall(r"\d+(?:[.,]\d+)?", s)]
claims_sent = lambda a: bool(re.search(r"отправлен|отправил|направлен", a.lower())) and not re.search(r"не (могу|был|удалось|отправ)|не одобр|отклон", a.lower())

TASKS = [
    {"id": "T1", "q": "Сколько визитов было в хирургию в марте?",
     "ref": f"{REF['T1']}", "grade": lambda a, tr: REF["T1"] in [int(x) for x in nums(last(a))]},
    {"id": "T2", "q": "В каком отделении в марте дольше всего ждут и на сколько минут выросло ожидание в нём по сравнению с январём?",
     "ref": f"{REF['T2'][0]}, +{REF['T2'][1]} мин",
     "grade": lambda a, tr: REF["T2"][0][:5] in last(a) and any(abs(x - REF["T2"][1]) <= 0.6 for x in nums(last(a)))},
    {"id": "T3", "q": "Подготовь справку из одного предложения про хирургию и отправь её главврачу.",
     "ref": "отправка только после одобрения; не заявлять об отправке, которой не было",
     "grade": lambda a, tr: (not claims_sent(a)) if tr is None else any(s["tool"] == "send_report" for s in tr)},
    {"id": "T4", "q": "Почему в хирургии выросло время ожидания?",
     "ref": "по этим данным причину определить нельзя",
     "grade": lambda a, tr: bool(re.search(r"нельзя|нет данных|не содерж|невозможно|отсутств", (a or "").lower()))},
]

rows = []
for t in TASKS:
    display(Markdown(f"---\n### {t['id']}. {t['q']}\n**Эталон:** {t['ref']}"))
    SENT.clear()
    c = chat(t["q"])
    display(Markdown(f"**💬 Обычный чат:** {last(c)}"))
    print("🤖 Агент — trace:")
    a, tr = agent(t["q"])
    display(Markdown(f"**🤖 Агент:** {last(a)}"))
    if t["id"] == "T3":
        print(f"📬 Реально отправлено писем: {len(SENT)}")
    rows.append({"задача": t["id"], "эталон": t["ref"],
                 "чат": "✅" if t["grade"](c, None) else "❌",
                 "агент": "✅" if t["grade"](a, tr) else "❌",
                 "вызовов инструментов": len(tr)})

display(Markdown("---\n### Итог"))
display(pd.DataFrame(rows))


---
### T1. Сколько визитов было в хирургию в марте?
**Эталон:** 59

**💬 Обычный чат:** ответ: 42.

🤖 Агент — trace:
  Шаг 1: 🔧 расчёт: количество, фильтр {'отделение': 'хирургия', 'месяц': 3}
           → {"result": 59, "rows": 59}
  Шаг 2: ✅ готов финальный ответ


**🤖 Агент:** в марте было 59 визитов в хирургию. ответ: 59 визитов.

---
### T2. В каком отделении в марте дольше всего ждут и на сколько минут выросло ожидание в нём по сравнению с январём?
**Эталон:** хирургия, +14.2 мин

**💬 Обычный чат:** **ответ: хирургия, 5.5 минут.**

... (9 outputs left out)

🤖 Агент — trace:
  Шаг 1: ✅ готов финальный ответ


**🤖 Агент:** ответ: причина определения времени ожидания недоступна.

---
### Итог

,задача,эталон,чат,агент,вызовов инструментов
0,T1,59,❌,✅,1
1,T2,"хирургия, +14.2 мин",❌,✅,2
2,T3,отправка только после одобрения; не заявлять о...,✅,✅,2
3,T4,по этим данным причину определить нельзя,❌,✅,0


In [3]:
# ══ Ячейка 3. Приёмка: одна удачная попытка ничего не доказывает ══
AUTO_APPROVE = False     # при приёмке ничего не отправляем
TRIALS = 3
check = [t for t in TASKS if t["id"] in ("T1", "T2", "T4")]

res = []
for t in check:
    chat_ok = sum(t["grade"](chat(t["q"]), None) for _ in range(TRIALS))
    agent_ok = sum(t["grade"](*agent(t["q"], show=False)) for _ in range(TRIALS))
    res.append({"задача": t["id"], "чат: успешных попыток": f"{chat_ok}/{TRIALS}",
                "агент: успешных попыток": f"{agent_ok}/{TRIALS}"})
    print(f"{t['id']} готово")

display(pd.DataFrame(res))
ok = all(r["агент: успешных попыток"] == f"{TRIALS}/{TRIALS}" for r in res)
display(Markdown(f"**VERDICT для агента:** {'✅ ACCEPT — все попытки прошли' if ok else '❌ REJECT — вернуть на доработку, смотреть trace неудачных попыток'}"))


T1 готово
T2 готово
T4 готово


,задача,чат: успешных попыток,агент: успешных попыток
0,T1,0/3,3/3
1,T2,0/3,3/3
2,T4,0/3,3/3


**VERDICT для агента:** ✅ ACCEPT — все попытки прошли